# MagScan-Mamba: pilot on the magnification-shift benchmark (BreakHis, benign vs malignant)

**Proposed model.** MagScan-Mamba classifies one histology tile at a magnification it never saw in training.

1. **Synthetic zoom pyramid.** From the single tile it builds three views: the full field, the centre 1/2
   and the centre 1/4, each resized to 224 x 224. A 2x zoom of a 200x tile covers roughly the field of a 400x
   tile, so training sees stand-ins for the unseen magnification.
2. **Frozen pathology foundation model.** Phikon (the most reliable encoder in the benchmark) turns each view
   into patch tokens; 14 x 14 tokens are average-pooled to 7 x 7, giving 3 x 49 tokens per tile.
3. **Scale scan.** A selective state-space (Mamba) mixer reads the token sequence along four paths: coarse to
   fine and fine to coarse, each row by row and column by column. The state carries the context of the coarser
   views into the finer ones. Two such layers, then mean pooling and a linear classifier.
4. **Reliability.** Temperature scaling on the validation set, and a Mahalanobis score on the pooled features
   that flags inputs unlike the training data (unseen magnification, another institution).

**What this pilot decides.** Before the full study is spent on it, MagScan-Mamba is compared with its own
ablations on exactly the benchmark's splits (same manifest, `patient_lomo_split`, seeds 13 / 42 / 2025), at
the two extreme held-out magnifications, 400x (where zooming in imitates the unseen magnification) and 40x
(where it cannot, since a tile cannot be zoomed out):

| Variant | Head | Views | Answers |
|---|---|---|---|
| `cls_probe` | linear on the CLS token | full view | reproduces the benchmark's Phikon linear probe |
| `cls3_probe` | linear on the three CLS tokens | 3 zooms | do the zoom views help without any token model? |
| `noscan` | the same block with the SSM removed (gated depthwise conv, MambaOut-style) | 3 zooms | is it the state-space scan that helps? |
| `attn` | the same block with self-attention as mixer | 3 zooms | Mamba or attention? |
| `magscan_single` | Mamba scan | full view only | is it the zoom pyramid that helps? |
| **`magscan`** | **Mamba scale scan** | **3 zooms** | **the proposed model** |

The decision rule is printed before the results: MagScan-Mamba passes if its mean macro-F1 beats `cls_probe`
at both held-out magnifications without worse ECE or risk at 80% coverage; the state-space claim holds only if
it also beats `noscan` and `attn`.

## Run guide (one Kaggle session, GPU T4, about 1-1.5 h)

1. **File -> Import Notebook**. Settings: **GPU T4**, **Internet ON**.
2. **Add Input:** `ambarish/breakhis` (required); `truthisneverlinear/bach-breast-cancer-histology-images`
   (optional, for the cross-institution score); the binary LOMO outputs of held-out 400x and 40x (optional:
   their records are shown next to the pilot, and the split is checked against their stored test labels).
3. **Save & Run All.** Phikon features for every tile and zoom are extracted once (about 10 min); every head
   then trains in about 1-2 min. If a commit stops early, attach this version as an input and run again:
   finished runs are restored and skipped.

In [ ]:
# ---------- Configuration. The split settings are the benchmark's and must not change ----------
HELD_OUT_MAGS = [400, 40]                 # pilot: the two extreme held-out magnifications
SEEDS = [13, 42, 2025]                    # the benchmark's seeds
VARIANTS = ['cls_probe', 'cls3_probe', 'noscan', 'attn', 'magscan_single', 'magscan']
RUN_NAME = 'magscan_pilot'
OUTPUT = f'/kaggle/working/{RUN_NAME}'
BREAKHIS_ROOT = None                      # None = auto-discover under /kaggle/input
BACH_ROOT = None

# ---- benchmark protocol (identical to the LOMO training notebooks) ----
MAGNIFICATIONS = [40, 100, 200, 400]      # the training pool; never narrowed
PATIENT_FRACTION = 1.0
IMAGE_SIZE = 224
TASK = 'binary'
NUM_CLASSES = 2
TEMPERATURE_SCALE = True
CHECKPOINT_RULE = 'maximum validation macro-F1; earliest epoch breaks ties; test evaluated once after selection'

# ---- MagScan-Mamba ----
FM_NAME = 'owkin/phikon'
ZOOMS = (1, 2, 4)                         # full field, centre 1/2, centre 1/4
TOKEN_POOL = 2                            # 14 x 14 patch tokens -> 7 x 7 per view
AUG_DRAWS = 2                             # augmented copies of every tile cached for training (plus 1 clean copy)
HEAD_DIM = 256
HEAD_LAYERS = 2
D_STATE = 16
D_CONV = 4
EXPAND = 2
ATTN_HEADS = 4
DROPOUT = 0.1
HEAD_EPOCHS = 30
HEAD_LR = 3e-4
HEAD_WEIGHT_DECAY = 0.05
HEAD_BATCH = 64
PROBE_EPOCHS = 200                        # the benchmark's linear-probe recipe
PROBE_LR = 1e-3
PROBE_WEIGHT_DECAY = 1e-4
PROBE_BATCH = 256
EXTRACT_BATCH = 32
NUM_WORKERS = 4
MAX_RUNTIME_HOURS = 11.5

assert set(HELD_OUT_MAGS) <= set(MAGNIFICATIONS)
assert set(VARIANTS) <= {'cls_probe', 'cls3_probe', 'noscan', 'attn', 'magscan_single', 'magscan'}
print(f'OUTPUT = {OUTPUT}\nheld-out {HELD_OUT_MAGS} x seeds {SEEDS} x variants {VARIANTS} '
      f'= {len(HELD_OUT_MAGS) * len(SEEDS) * len(VARIANTS)} runs')

In [ ]:
# ---------- Environment: packages, the selective-scan kernel, and a check of that kernel ----------
import os, sys, json, math, random, time, hashlib, re, shutil, subprocess, platform
from pathlib import Path
from datetime import datetime, timezone

Path(OUTPUT).mkdir(parents=True, exist_ok=True)


def pip_install_if_missing(module_name, spec=None):
    try:
        __import__(module_name); return True
    except ImportError:
        try:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', spec or module_name], check=True)
            return True
        except subprocess.CalledProcessError as exc:
            print(f'Install failed for {spec or module_name}:', exc); return False


for _m, _s in [('transformers', 'transformers'), ('einops', 'einops')]:
    pip_install_if_missing(_m, _s)

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import torchvision, sklearn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                             average_precision_score, roc_curve)
from sklearn.covariance import LedoitWolf

assert torch.cuda.is_available(), 'Kaggle GPU accelerator is required (Settings -> Accelerator -> GPU T4).'
DEVICE = torch.device('cuda:0')

# The compiled selective scan of mamba_ssm (the same kernel MedMamba uses). Only selective_scan_fn is needed,
# so causal-conv1d is optional. Without the kernel the pure-torch recurrence below is used: same numbers,
# about 10-20x slower.
SCAN_BACKEND = 'reference'
try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn as _cuda_scan
    SCAN_BACKEND = 'cuda'
except Exception:
    # the specs that installed MedMamba's kernel in the benchmark notebooks, in the same order
    for _spec in [['mamba-ssm', 'causal-conv1d'], ['mamba-ssm==2.2.4', 'causal-conv1d==1.5.0.post8'],
                  ['mamba-ssm==2.2.2', 'causal-conv1d==1.4.0']]:
        try:
            print('Installing', _spec)
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation'] + _spec, check=True)
            from mamba_ssm.ops.selective_scan_interface import selective_scan_fn as _cuda_scan
            SCAN_BACKEND = 'cuda'; break
        except Exception as exc:
            print('  failed:', str(exc)[:160])


def selective_scan_reference(u, delta, A, B, C, D=None, z=None, delta_bias=None, delta_softplus=False):
    """Pure-torch selective scan (the recurrence of Gu & Dao, 2024).
    u, delta: (b, d, l); A: (d, n); B, C: (b, g, n, l) with d divisible by g; D: (d,)."""
    dtype_in = u.dtype
    u, delta = u.float(), delta.float()
    if delta_bias is not None:
        delta = delta + delta_bias[..., None].float()
    if delta_softplus:
        delta = F.softplus(delta)
    b, d, l = u.shape
    B = B.float().repeat_interleave(d // B.shape[1], dim=1)              # (b, d, n, l)
    C = C.float().repeat_interleave(d // C.shape[1], dim=1)
    dA = torch.exp(torch.einsum('bdl,dn->bdln', delta, A.float()))       # (b, d, l, n)
    dBu = torch.einsum('bdl,bdnl,bdl->bdln', delta, B, u)                # (b, d, l, n)
    h = u.new_zeros(b, d, A.shape[1])
    ys = []
    for i in range(l):
        h = dA[:, :, i] * h + dBu[:, :, i]
        ys.append(torch.einsum('bdn,bdn->bd', h, C[:, :, :, i]))
    y = torch.stack(ys, dim=2)
    if D is not None:
        y = y + u * D.float()[..., None]
    if z is not None:
        y = y * F.silu(z.float())
    return y.to(dtype_in)


SELECTIVE_SCAN = _cuda_scan if SCAN_BACKEND == 'cuda' else selective_scan_reference
if SCAN_BACKEND == 'cuda':
    # The kernel and the reference must agree before any result is trusted.
    _g = torch.Generator().manual_seed(0)
    _b, _d, _l, _n = 2, 8, 37, 4
    _u, _dt = torch.randn(_b, _d, _l, generator=_g), torch.randn(_b, _d, _l, generator=_g)
    _A = -torch.rand(_d, _n, generator=_g) - 0.1
    _B, _C = torch.randn(_b, 1, _n, _l, generator=_g), torch.randn(_b, 1, _n, _l, generator=_g)
    _D, _bias = torch.randn(_d, generator=_g), torch.randn(_d, generator=_g)
    _ref = selective_scan_reference(_u, _dt, _A, _B, _C, _D, delta_bias=_bias, delta_softplus=True)
    _ker = _cuda_scan(*[t.to(DEVICE) for t in (_u, _dt, _A, _B, _C, _D)], z=None, delta_bias=_bias.to(DEVICE),
                      delta_softplus=True).cpu()
    _diff = float((_ref - _ker).abs().max())
    assert _diff < 1e-3, f'CUDA selective scan disagrees with the reference (max diff {_diff:.2e})'
    print(f'selective scan: CUDA kernel, matches the reference (max diff {_diff:.1e})')
else:
    print('selective scan: pure-torch reference (mamba_ssm unavailable); results are the same, only slower')

env = {'timestamp_utc': datetime.now(timezone.utc).isoformat(), 'python': sys.version, 'torch': torch.__version__,
       'torch_cuda': torch.version.cuda, 'gpu_name': torch.cuda.get_device_name(0), 'torchvision': torchvision.__version__,
       'sklearn': sklearn.__version__, 'scan_backend': SCAN_BACKEND}
Path(OUTPUT, 'environment.json').write_text(json.dumps(env, indent=2))
Path(OUTPUT, 'config.json').write_text(json.dumps({k: v for k, v in globals().items()
                                                   if k.isupper() and k not in {'OUTPUT', 'DEVICE'}},
                                                  indent=2, default=str))
print(json.dumps(env, indent=2))

In [ ]:
# ---------- BreakHis manifest and the patient-disjoint LOMO split: copied verbatim from the benchmark ----------
# Same file-name pattern, the same row order (sorted paths, no de-duplication) and the same patient_lomo_split,
# so seed s at held-out magnification m gives exactly the benchmark's train / val / test patients. The manifest
# hash is compared with the one stored by any attached benchmark output.
NAME = re.compile(r'^(?P<patient>SOB_(?P<coarse>[BM])_(?P<subtype>[A-Z]+)-\d+-[A-Za-z0-9]+)-(?P<magnification>\d+)-\d+', re.I)
SUBTYPE_TO_ID = {'A': 0, 'F': 1, 'PT': 2, 'TA': 3, 'DC': 4, 'LC': 5, 'MC': 6, 'PC': 7}


def discover_breakhis_root(search_root='/kaggle/input'):
    candidates = []
    if Path(search_root).exists():
        candidates = [top for top in sorted(Path(search_root).iterdir()) if top.is_dir()] + [Path(search_root)]
    best, best_count = None, 0
    for candidate in candidates:
        try:
            count = sum(1 for _ in Path(candidate).rglob('SOB_*.png'))
        except (PermissionError, OSError):
            continue
        if count > best_count:
            best, best_count = candidate, count
    if best is None or best_count == 0:
        raise RuntimeError('Could not auto-discover BreakHis root. Attach ambarish/breakhis or set BREAKHIS_ROOT.')
    print('Auto-discovered BREAKHIS_ROOT =', best, ' (', best_count, 'SOB PNG files)')
    return str(best)


def build_manifest(root):
    rows = []
    for path in sorted(Path(root).rglob('SOB_*.png')):
        match = NAME.match(path.name)
        if not match:
            continue
        item = match.groupdict()
        subtype = item['subtype'].upper()
        if subtype not in SUBTYPE_TO_ID:
            continue
        rows.append({'path': str(path), 'patient_id': item['patient'], 'magnification': int(item['magnification']),
                     'coarse_label': int(item['coarse'].upper() == 'M'), 'subtype': subtype,
                     'subtype_label': SUBTYPE_TO_ID[subtype]})
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise RuntimeError('No matching images found under ' + str(root))
    return frame


def patient_lomo_split(frame, held_out_mag, fraction=1.0, seed=2025,
                       val_ratio=0.15, test_ratio=0.20, val_on_held_out=False):
    """Patient-disjoint LOMO split with per-class stratification (the benchmark's function, unchanged)."""
    frame = frame.copy()
    patient_class = frame.groupby('patient_id').coarse_label.first().reset_index()
    rng = np.random.default_rng(seed)
    train_patients, val_patients, test_patients = set(), set(), set()
    for cls in sorted(patient_class.coarse_label.unique()):
        group = patient_class.loc[patient_class.coarse_label == cls, 'patient_id'].to_numpy()
        perm = rng.permutation(group)
        n = len(perm)
        n_test = max(1, int(round(test_ratio * n)))
        n_val = max(1, int(round(val_ratio * n)))
        assert n_test + n_val < n, 'Class ' + str(cls) + ' has only ' + str(n) + ' patients; val+test ratios too high.'
        test_patients.update(perm[:n_test].tolist())
        val_patients.update(perm[n_test:n_test + n_val].tolist())
        remain = perm[n_test + n_val:].tolist()
        n_keep = max(1, int(round(fraction * len(remain))))
        train_patients.update(remain[:n_keep])
    train_mags = [m for m in MAGNIFICATIONS if m != held_out_mag]
    frame['split'] = 'drop'
    frame.loc[frame.patient_id.isin(train_patients) & frame.magnification.isin(train_mags), 'split'] = 'train'
    if val_on_held_out:
        frame.loc[frame.patient_id.isin(val_patients) & (frame.magnification == held_out_mag), 'split'] = 'val'
    else:
        frame.loc[frame.patient_id.isin(val_patients) & frame.magnification.isin(train_mags), 'split'] = 'val'
    frame.loc[frame.patient_id.isin(test_patients) & (frame.magnification == held_out_mag), 'split'] = 'test'
    subset = frame.loc[frame.split != 'drop'].copy()
    leak = subset.groupby('patient_id').split.nunique().gt(1).any()
    assert not leak, 'Patient leakage detected.'
    for name in ('train', 'val', 'test'):
        rows = subset[subset.split == name]
        if len(rows) == 0:
            raise RuntimeError('Split ' + name + ' empty (held_out=' + str(held_out_mag) + ', seed=' + str(seed) + ').')
        if rows.coarse_label.nunique() < 2:
            raise RuntimeError('Split ' + name + ' has only one class -- stratification failed.')
    return subset


if BREAKHIS_ROOT is None or not list(Path(BREAKHIS_ROOT).rglob('SOB_*.png')):
    BREAKHIS_ROOT = discover_breakhis_root()
MANIFEST = build_manifest(BREAKHIS_ROOT)
MANIFEST.to_csv(Path(OUTPUT, 'breakhis_manifest.csv'), index=False)
MANIFEST_HASH = hashlib.sha256(MANIFEST.to_csv(index=False).encode()).hexdigest()      # the benchmark's hash (with paths)
# The same rows without the mount point: identifies the data across sessions wherever the dataset is mounted.
MANIFEST_KEY = hashlib.sha256(MANIFEST.assign(path=MANIFEST.path.map(lambda p: Path(p).name))
                              .to_csv(index=False).encode()).hexdigest()
print('total images:', len(MANIFEST), '  patients:', MANIFEST.patient_id.nunique(), '  manifest SHA-256:', MANIFEST_HASH)

# ---- attached benchmark outputs: their manifest hash and runs (for the split check and the reference rows) ----
BENCHMARK_DIRS = []
if Path('/kaggle/input').exists():
    BENCHMARK_DIRS = sorted({p.parent.parent.parent for p in Path('/kaggle/input').rglob('record.json')
                             if p.parent.parent.name == 'runs'
                             and p.parent.name.startswith(('scratch_', 'fm_owkin_phikon_'))})
_hashes = {str(d): Path(d, 'breakhis_manifest.sha256').read_text().strip()
           for d in BENCHMARK_DIRS if Path(d, 'breakhis_manifest.sha256').exists()}
for d, h in _hashes.items():
    # The benchmark notebooks hash their manifest CSV the same way, but their path column holds their own
    # mount point, so a differing hash is expected when the dataset is mounted elsewhere; the test-label
    # check in the training cell is the decisive one.
    print(f'  benchmark output {d}: manifest hash {"identical" if h == MANIFEST_HASH else "differs (mount path)"}')
print(f'{len(BENCHMARK_DIRS)} benchmark output folder(s) attached' + ('' if BENCHMARK_DIRS else
      ' - the pilot runs anyway; attach them to see the reference rows and the split check'))

In [ ]:
# ---------- Metrics: the benchmark's calibration / selective-prediction code and section 2g's OOD scores ----------
_trapz = getattr(np, 'trapezoid', None) or np.trapz


def expected_calibration_error(probs, labels, n_bins=15):
    confidences = probs.max(axis=1); predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for lower, upper in zip(bins[:-1], bins[1:]):
        in_bin = (confidences > lower) & (confidences <= upper)
        if in_bin.sum() == 0: continue
        ece += in_bin.mean() * abs(accuracies[in_bin].mean() - confidences[in_bin].mean())
    return float(ece)


def brier_multiclass(probs, labels):
    return float(((probs - np.eye(probs.shape[1])[labels]) ** 2).sum(axis=1).mean())


class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__(); self.log_temp = nn.Parameter(torch.zeros(1))

    def forward(self, logits):
        return logits / torch.exp(self.log_temp)


def fit_temperature(val_logits, val_labels, max_iter=200):
    scaler = TemperatureScaler().to(DEVICE)
    logits = torch.as_tensor(val_logits, device=DEVICE, dtype=torch.float32)
    labels = torch.as_tensor(val_labels, device=DEVICE, dtype=torch.long)
    optim = torch.optim.LBFGS([scaler.log_temp], lr=0.1, max_iter=max_iter)
    loss_fn = nn.CrossEntropyLoss()

    def closure():
        optim.zero_grad(); loss = loss_fn(scaler(logits), labels); loss.backward(); return loss
    optim.step(closure)
    return float(torch.exp(scaler.log_temp).item())


def risk_coverage_curve(probs, labels):
    confidences = probs.max(axis=1); predictions = probs.argmax(axis=1)
    order = np.argsort(-confidences)
    errors = (predictions != labels).astype(float)[order]
    coverage = np.arange(1, len(order) + 1) / len(order)
    risk = np.cumsum(errors) / np.arange(1, len(order) + 1)
    return coverage, risk, float(_trapz(risk, coverage))


def compute_all_metrics(logits, labels, temperature=1.0):
    scaled = logits / temperature
    exp = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    probs = exp / exp.sum(axis=1, keepdims=True)
    predictions = probs.argmax(axis=1)
    metrics = {'accuracy': accuracy_score(labels, predictions),
               'balanced_accuracy': balanced_accuracy_score(labels, predictions),
               'macro_f1': f1_score(labels, predictions, average='macro'),
               'ece': expected_calibration_error(probs, labels), 'brier': brier_multiclass(probs, labels)}
    try:
        metrics['auc'] = roc_auc_score(labels, probs[:, 1])
    except ValueError:
        metrics['auc'] = float('nan')
    coverage, risk, rcauc = risk_coverage_curve(probs, labels)
    metrics['risk_coverage_auc'] = rcauc
    for target in (0.80, 0.90, 0.95):
        idx = int(target * len(coverage)) - 1
        metrics['risk_at_coverage_' + str(int(100 * target))] = float(risk[idx]) if idx >= 0 else float('nan')
    # E-AURC (Geifman et al., ICLR 2019): AURC minus its lowest value at this error rate
    r = min(1.0 - metrics['accuracy'], 1 - 1e-12)
    metrics['e_aurc'] = rcauc - (r + (1 - r) * np.log(1 - r))
    metrics['temperature'] = float(temperature)
    return metrics, probs


# ---- OOD scores, as in the analysis notebook's section 2g (higher = more out-of-distribution) ----
def energy_from_logits(logits):
    z = np.asarray(logits, dtype=np.float64); m = z.max(axis=1, keepdims=True)
    return -(m[:, 0] + np.log(np.exp(z - m).sum(axis=1)))


def neg_msp(logits):
    z = np.asarray(logits, dtype=np.float64); z = z - z.max(axis=1, keepdims=True)
    p = np.exp(z); p /= p.sum(axis=1, keepdims=True)
    return -p.max(axis=1)


def fit_mahalanobis(train_feats, train_labels):
    """Class means + one Ledoit-Wolf tied covariance on class-centred training features."""
    X = np.asarray(train_feats, dtype=np.float64); y = np.asarray(train_labels).astype(int)
    classes = np.unique(y)
    means = np.stack([X[y == c].mean(axis=0) for c in classes])
    precision = LedoitWolf(assume_centered=True).fit(X - means[np.searchsorted(classes, y)]).precision_
    return means, precision


def mahalanobis_score(feats, means, precision):
    X = np.asarray(feats, dtype=np.float64)
    return np.min(np.stack([np.einsum('nd,nd->n', (X - mu) @ precision, X - mu) for mu in means], axis=1), axis=1)


def knn_score(feats, train_feats, k=10, chunk=2048):
    def _n(a):
        a = np.asarray(a, dtype=np.float64); return a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-12)
    bank, q = _n(train_feats), _n(feats)
    k = int(min(k, len(bank))); out = np.empty(len(q))
    for s in range(0, len(q), chunk):
        sims = q[s:s + chunk] @ bank.T
        kth = -np.partition(-sims, k - 1, axis=1)[:, k - 1]
        out[s:s + chunk] = np.sqrt(np.clip(2.0 - 2.0 * kth, 0.0, None))
    return out


def ood_auroc(id_scores, ood_scores):
    y = np.r_[np.zeros(len(id_scores)), np.ones(len(ood_scores))]
    return float(roc_auc_score(y, np.r_[id_scores, ood_scores]))


print('metrics ready')

In [ ]:
# ---------- Frozen Phikon tokens for every tile, zoom view and augmentation draw (extracted once) ----------
# Draw 0 is the clean tile (validation, test, the Mahalanobis reference); draws 1..AUG_DRAWS apply the
# benchmark's training augmentation (flips, rotation up to 15 degrees, colour jitter) to the full tile before it
# is cut into zoom views, so the three views of a draw show the same augmented tissue. Each draw is seeded by
# (draw, tile), so it is identical in every session. Stored per tile: (zoom, 1 CLS + 49 pooled patch tokens, 768)
# in float16; Phikon itself runs in float32, as in the benchmark.
from transformers import AutoModel

_NORMALIZE = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(),
                                 transforms.Normalize((.485, .456, .406), (.229, .224, .225))])
_TRAIN_AUG = transforms.Compose([transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                                 transforms.RandomRotation(15), transforms.ColorJitter(.1, .1, .05)])


def zoom_views(img):
    """(Z, 3, 224, 224): centre crops of 1/z of the tile's width and height, each resized to 224 x 224.
    Zoom 1 is the whole tile, i.e. exactly the benchmark's input."""
    w, h = img.size
    views = []
    for z in ZOOMS:
        cw, ch = max(1, round(w / z)), max(1, round(h / z))
        left, top = (w - cw) // 2, (h - ch) // 2
        views.append(_NORMALIZE(img.crop((left, top, left + cw, top + ch))))
    return torch.stack(views)


class ZoomDataset(Dataset):
    def __init__(self, paths, draw):
        self.paths, self.draw = list(paths), int(draw)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        if self.draw > 0:
            torch.manual_seed(1_000_003 * self.draw + i)
            img = _TRAIN_AUG(img)
        return zoom_views(img), i


FM = AutoModel.from_pretrained(FM_NAME, token=os.environ.get('HF_TOKEN'), trust_remote_code=True).eval().to(DEVICE)
for _p in FM.parameters():
    _p.requires_grad_(False)


@torch.inference_mode()
def extract_tokens(paths, draw):
    loader = DataLoader(ZoomDataset(paths, draw), batch_size=EXTRACT_BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
    out = None
    for views, idx in loader:
        b, z = views.shape[:2]
        h = FM(pixel_values=views.flatten(0, 1).to(DEVICE, non_blocking=True)).last_hidden_state   # (b*z, 1+g*g, D)
        g = int(round((h.shape[1] - 1) ** 0.5))
        assert g * g == h.shape[1] - 1, f'unexpected token count {h.shape[1]}'
        grid = h[:, 1:].reshape(b * z, g, g, -1).permute(0, 3, 1, 2)                           # (b*z, D, g, g)
        pooled = F.avg_pool2d(grid, TOKEN_POOL).flatten(2).transpose(1, 2)                     # (b*z, (g/p)^2, D)
        tok = torch.cat([h[:, :1], pooled], dim=1).reshape(b, z, -1, h.shape[-1])
        if out is None:
            out = torch.empty((len(loader.dataset),) + tuple(tok.shape[1:]), dtype=torch.float16)
        out[idx] = tok.half().cpu()
    return out


t0 = time.perf_counter()
TOKENS = {d: extract_tokens(MANIFEST.path.tolist(), d) for d in range(AUG_DRAWS + 1)}
LABELS = MANIFEST.coarse_label.to_numpy().astype(int)
N_TOKENS_PER_VIEW = TOKENS[0].shape[2] - 1
FM_DIM = TOKENS[0].shape[-1]
print(f'BreakHis tokens: {len(TOKENS)} draws x {tuple(TOKENS[0].shape)} float16 '
      f'({sum(t.numel() for t in TOKENS.values()) * 2 / 2**30:.1f} GiB) in {(time.perf_counter() - t0) / 60:.1f} min')

# Zoom 1 of draw 0 must be the benchmark's own Phikon input: compare its CLS with a direct forward pass.
with torch.inference_mode():
    _x = torch.stack([_NORMALIZE(Image.open(p).convert('RGB')) for p in MANIFEST.path[:4]]).to(DEVICE)
    _cls = FM(pixel_values=_x).last_hidden_state[:, 0].float().cpu()
_d = float((_cls - TOKENS[0][:4, 0, 0].float()).abs().max() / _cls.abs().max())
assert _d < 5e-3, f'zoom-1 CLS differs from the benchmark input (relative {_d:.1e})'
print(f'zoom-1 CLS = the benchmark\'s Phikon feature (relative diff {_d:.1e}, float16 storage)')


# ---- BACH (another institution), clean views only ----
def discover_bach_root(search_root='/kaggle/input'):
    best, best_count = None, 0
    if Path(search_root).exists():
        for root, dirs, _files in os.walk(search_root):
            if {'Normal', 'Benign', 'InSitu', 'Invasive'}.issubset(set(dirs)):
                count = sum(1 for _ in Path(root).rglob('*.tif')) + sum(1 for _ in Path(root).rglob('*.png'))
                if count > best_count:
                    best, best_count = Path(root), count
    if best is not None:
        print(f'BACH_ROOT = {best} ({best_count} image files)')
    return None if best is None else str(best)


BACH_TOKENS = None
if BACH_ROOT is None:
    BACH_ROOT = discover_bach_root()
if BACH_ROOT is not None:
    _bach_paths = [str(p) for cls in ('Normal', 'Benign', 'InSitu', 'Invasive')
                   for sfx in ('*.tif', '*.png', '*.jpg') for p in sorted(Path(BACH_ROOT, cls).glob(sfx))]
    BACH_TOKENS = extract_tokens(_bach_paths, 0)
    print(f'BACH tokens: {tuple(BACH_TOKENS.shape)}')
else:
    print('BACH not attached: the cross-institution score is skipped')

In [ ]:
# ---------- MagScan-Mamba and its ablations ----------
# Input per tile: (Z, 1 + 49, 768) Phikon tokens. The token heads use the 49 pooled patch tokens of each view;
# the probes use the CLS tokens. All token heads share everything except the token mixer:
#   LayerNorm -> Linear(768, D) + scale embedding + 2-D position embedding
#   -> HEAD_LAYERS x [ x + mixer(LN(x)) ; x + MLP(LN(x)) ] -> LN -> mean over tokens -> dropout -> Linear(D, 2)
# mixer = 'mamba' : one selective-SSM mixer (Mamba block) run along four orders of the (zoom, row, col) grid -
#                   coarse->fine row-major, its reverse, coarse->fine column-major, its reverse - mapped back
#                   to the original order and averaged. Weights are shared across the four orders.
#         'conv'  : the same block with the SSM removed: gated causal depthwise conv only (MambaOut, CVPR 2025).
#         'attn'  : multi-head self-attention over all tokens.
class MambaMixer(nn.Module):
    """Mamba (S6) block of Gu & Dao (2024): in_proj -> causal depthwise conv -> SiLU -> selective scan,
    gated by SiLU(z) -> out_proj. use_ssm=False drops the selective scan and keeps the gated conv."""
    def __init__(self, d_model, d_state=D_STATE, d_conv=D_CONV, expand=EXPAND, use_ssm=True):
        super().__init__()
        self.d_inner, self.d_state, self.use_ssm = expand * d_model, d_state, use_ssm
        self.dt_rank = math.ceil(d_model / 16)
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, d_conv, groups=self.d_inner, padding=d_conv - 1)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        if use_ssm:
            self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * d_state, bias=False)
            self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)
            nn.init.uniform_(self.dt_proj.weight, -self.dt_rank ** -0.5, self.dt_rank ** -0.5)
            dt = torch.exp(torch.rand(self.d_inner) * (math.log(0.1) - math.log(0.001)) + math.log(0.001)).clamp(min=1e-4)
            with torch.no_grad():
                self.dt_proj.bias.copy_(dt + torch.log(-torch.expm1(-dt)))          # softplus^-1(dt), as in Mamba
            self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)))
            self.D = nn.Parameter(torch.ones(self.d_inner))

    def forward(self, x):                                                          # x: (B, L, d_model)
        L = x.shape[1]
        u, z = self.in_proj(x).chunk(2, dim=-1)                                    # (B, L, d_inner) each
        u = F.silu(self.conv1d(u.transpose(1, 2))[..., :L])                        # (B, d_inner, L), causal
        if self.use_ssm:
            dt, Bm, Cm = torch.split(self.x_proj(u.transpose(1, 2)), [self.dt_rank, self.d_state, self.d_state], dim=-1)
            dt = (dt @ self.dt_proj.weight.t()).transpose(1, 2)                   # (B, d_inner, L); bias in the scan
            u = SELECTIVE_SCAN(u.float().contiguous(), dt.float().contiguous(), -torch.exp(self.A_log.float()),
                               Bm.transpose(1, 2).unsqueeze(1).float().contiguous(),   # (B, 1, N, L)
                               Cm.transpose(1, 2).unsqueeze(1).float().contiguous(),
                               self.D.float(), z=None, delta_bias=self.dt_proj.bias.float(), delta_softplus=True)
        return self.out_proj(u.transpose(1, 2) * F.silu(z))


def scan_orders(n_zoom, grid):
    """Four index orders over tokens stored as (zoom, row, col); each maps back with argsort."""
    idx = torch.arange(n_zoom * grid * grid).reshape(n_zoom, grid, grid)
    row_major = idx.flatten()
    col_major = idx.transpose(1, 2).flatten()
    orders = [row_major, row_major.flip(0), col_major, col_major.flip(0)]
    return [(o, torch.argsort(o)) for o in orders]


class ScanMixer(nn.Module):
    def __init__(self, d_model, n_zoom, grid, use_ssm=True):
        super().__init__()
        self.core = MambaMixer(d_model, use_ssm=use_ssm)
        for k, (o, inv) in enumerate(scan_orders(n_zoom, grid)):
            self.register_buffer(f'order{k}', o, persistent=False)
            self.register_buffer(f'inverse{k}', inv, persistent=False)

    def forward(self, x):
        out = 0
        for k in range(4):
            o, inv = getattr(self, f'order{k}'), getattr(self, f'inverse{k}')
            out = out + self.core(x[:, o])[:, inv]
        return out / 4


class AttnMixer(nn.Module):
    def __init__(self, d_model):
        super().__init__(); self.attn = nn.MultiheadAttention(d_model, ATTN_HEADS, dropout=DROPOUT, batch_first=True)

    def forward(self, x):
        return self.attn(x, x, x, need_weights=False)[0]


class TokenHead(nn.Module):
    def __init__(self, mixer, n_zoom, grid, in_dim, d=HEAD_DIM, layers=HEAD_LAYERS, num_classes=NUM_CLASSES):
        super().__init__()
        self.n_zoom, self.grid = n_zoom, grid
        self.inp = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, d))
        self.scale_emb = nn.Parameter(torch.zeros(1, n_zoom, 1, d))
        self.pos_emb = nn.Parameter(torch.zeros(1, 1, grid * grid, d))
        nn.init.trunc_normal_(self.scale_emb, std=0.02); nn.init.trunc_normal_(self.pos_emb, std=0.02)
        make = {'mamba': lambda: ScanMixer(d, n_zoom, grid, True), 'conv': lambda: ScanMixer(d, n_zoom, grid, False),
                'attn': lambda: AttnMixer(d)}[mixer]
        self.blocks = nn.ModuleList([nn.ModuleDict({'n1': nn.LayerNorm(d), 'mix': make(), 'n2': nn.LayerNorm(d),
                                                    'mlp': nn.Sequential(nn.Linear(d, 2 * d), nn.GELU(), nn.Dropout(DROPOUT),
                                                                         nn.Linear(2 * d, d))})
                                     for _ in range(layers)])
        self.norm, self.drop, self.fc = nn.LayerNorm(d), nn.Dropout(DROPOUT), nn.Linear(d, num_classes)

    def features(self, tokens):                            # tokens: (B, Z, 1 + grid^2, in_dim) -> (B, d)
        x = self.inp(tokens[:, :, 1:].float()) + self.scale_emb + self.pos_emb     # (B, Z, grid^2, d)
        x = x.flatten(1, 2)                                                         # (B, Z * grid^2, d): (zoom, row, col)
        for blk in self.blocks:
            x = x + blk['mix'](blk['n1'](x))
            x = x + blk['mlp'](blk['n2'](x))
        return self.norm(x).mean(dim=1)

    def forward(self, tokens):
        return self.fc(self.drop(self.features(tokens)))


class ClsProbe(nn.Module):
    """Linear probe on the CLS token of each used view (one view = the benchmark's Phikon probe)."""
    def __init__(self, n_zoom, in_dim, num_classes=NUM_CLASSES):
        super().__init__(); self.fc = nn.Linear(n_zoom * in_dim, num_classes)

    def features(self, tokens):
        return tokens[:, :, 0].float().flatten(1)

    def forward(self, tokens):
        return self.fc(self.features(tokens))


VARIANT_SPEC = {   # variant: (kind, zoom indices into ZOOMS)
    'cls_probe': ('probe', [0]), 'cls3_probe': ('probe', list(range(len(ZOOMS)))),
    'noscan': ('conv', list(range(len(ZOOMS)))), 'attn': ('attn', list(range(len(ZOOMS)))),
    'magscan_single': ('mamba', [0]), 'magscan': ('mamba', list(range(len(ZOOMS))))}
GRID = int(round(N_TOKENS_PER_VIEW ** 0.5))


def build_variant(variant):
    kind, zooms = VARIANT_SPEC[variant]
    if kind == 'probe':
        return ClsProbe(len(zooms), FM_DIM)
    return TokenHead(kind, len(zooms), GRID, FM_DIM)


# ---- self-checks: orders invert, shapes, and the parameter budget of every head ----
for o, inv in scan_orders(3, GRID):
    assert torch.equal(o[inv], torch.arange(len(o))), 'scan order does not invert'
_rows = []
for v in VARIANTS:
    m = build_variant(v).to(DEVICE)
    zs = VARIANT_SPEC[v][1]
    with torch.no_grad():
        out = m(torch.randn(2, len(zs), 1 + GRID * GRID, FM_DIM, device=DEVICE))
    assert out.shape == (2, NUM_CLASSES)
    _rows.append({'variant': v, 'trainable_params': sum(p.numel() for p in m.parameters() if p.requires_grad),
                  'tokens': len(zs) * GRID * GRID if VARIANT_SPEC[v][0] != 'probe' else len(zs)})
    del m
PARAMS = pd.DataFrame(_rows).set_index('variant')
print(PARAMS.to_string())

In [ ]:
# ---------- Train and evaluate one variant at one (held-out magnification, seed) ----------
# Split: the benchmark's patient_lomo_split. Training tiles use a random augmented draw each epoch; validation,
# test and the Mahalanobis reference use the clean draw. Checkpoint: best validation macro-F1 (earliest epoch on
# ties), then temperature scaling on the validation logits, then one test evaluation - the benchmark's rule.
# Probes follow the benchmark's probe recipe (AdamW 1e-3, 200 epochs, batch 256); token heads use AdamW 3e-4
# with cosine decay for 30 epochs, batch 64. The loss is unweighted cross-entropy, as in the benchmark.
import copy

CONFIG_KEYS = ['MANIFEST_KEY', 'ZOOMS', 'TOKEN_POOL', 'AUG_DRAWS', 'HEAD_DIM', 'HEAD_LAYERS', 'D_STATE', 'D_CONV',
               'EXPAND', 'ATTN_HEADS', 'DROPOUT', 'HEAD_EPOCHS', 'HEAD_LR', 'HEAD_WEIGHT_DECAY', 'HEAD_BATCH',
               'PROBE_EPOCHS', 'PROBE_LR', 'PROBE_WEIGHT_DECAY', 'PROBE_BATCH', 'FM_NAME', 'PATIENT_FRACTION']
CONFIG_HASH = hashlib.sha256(json.dumps({k: globals()[k] for k in CONFIG_KEYS}, default=str,
                                        sort_keys=True).encode()).hexdigest()[:16]
OOD_PAIRS = {'test_vs_val': ('val', 'test'), 'bach_vs_test': ('test', 'bach')}


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


def run_id_for(variant, ho, seed):
    return f'magscan_{variant}_ho{ho}_frac{PATIENT_FRACTION}_seed{seed}'


def split_indices(ho, seed):
    sub = patient_lomo_split(MANIFEST, ho, fraction=PATIENT_FRACTION, seed=seed)
    return {s: np.array(sub.index[sub.split == s]) for s in ('train', 'val', 'test')}      # manifest rows, in order


def benchmark_split_check(ho, seed, test_rows):
    """Compare the test labels with the stored ones of the benchmark's Phikon probe run (same order)."""
    rid = f'fm_owkin_phikon_ho{ho}_frac{PATIENT_FRACTION}_seed{seed}'
    for d in BENCHMARK_DIRS:
        f = Path(d, 'runs', rid, 'test_outputs.npz')
        if f.exists():
            stored = np.load(f)['labels']
            return 'identical' if np.array_equal(stored, LABELS[test_rows]) else 'DIFFERS'
    return 'benchmark run not attached'


@torch.no_grad()
def predict(model, X, batch=256):
    model.eval()
    logits, feats = [], []
    for s in range(0, len(X), batch):
        xb = X[s:s + batch].to(DEVICE)
        f = model.features(xb)
        feats.append(f.float().cpu()); logits.append(model.fc(f).float().cpu())
    return torch.cat(logits).numpy(), torch.cat(feats).numpy()


def run_variant(variant, ho, seed):
    seed_everything(seed)
    kind, zooms = VARIANT_SPEC[variant]
    idx = split_indices(ho, seed)
    zsel = torch.as_tensor(zooms)
    token_slice = slice(0, 1) if kind == 'probe' else slice(None)        # probes need only the CLS token

    def gather(draw, rows):
        return TOKENS[draw][torch.as_tensor(rows)][:, zsel][:, :, token_slice]

    draws = list(range(1, AUG_DRAWS + 1)) or [0]
    X_train = torch.stack([gather(d, idx['train']) for d in draws]).to(DEVICE)   # (draws, n, Z, T, D) float16
    X = {s: gather(0, idx[s]) for s in ('train', 'val', 'test')}                   # clean, on the CPU
    y = {s: LABELS[idx[s]] for s in ('train', 'val', 'test')}
    y_train_dev = torch.as_tensor(y['train'], device=DEVICE)

    model = build_variant(variant).to(DEVICE)
    probe = kind == 'probe'
    epochs, lr, wd, bs = ((PROBE_EPOCHS, PROBE_LR, PROBE_WEIGHT_DECAY, PROBE_BATCH) if probe else
                          (HEAD_EPOCHS, HEAD_LR, HEAD_WEIGHT_DECAY, HEAD_BATCH))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    n = len(y['train']); steps = math.ceil(n / bs)
    sched = None if probe else torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * steps)
    loss_fn = nn.CrossEntropyLoss()
    gen = torch.Generator().manual_seed(seed)
    best, best_state, best_epoch, log = -1.0, None, 0, []
    torch.cuda.reset_peak_memory_stats(); start = time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train(); running = 0.0
        order = torch.randperm(n, generator=gen)
        for s in range(0, n, bs):
            b = order[s:s + bs]
            d = torch.randint(0, len(draws), (len(b),), generator=gen)              # one draw per tile per epoch
            xb = X_train[d.to(DEVICE), b.to(DEVICE)]
            loss = loss_fn(model(xb), y_train_dev[b.to(DEVICE)])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            if sched is not None:
                sched.step()
            running += float(loss.detach()) * len(b)
        vl, _ = predict(model, X['val'])
        vm, _ = compute_all_metrics(vl, y['val'])
        log.append({'epoch': epoch, 'train_loss': running / n, **{f'val_{k}': v for k, v in vm.items()}})
        if vm['macro_f1'] > best:                                                   # strict: earliest epoch wins ties
            best, best_epoch = vm['macro_f1'], epoch
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    del X_train
    val_logits, val_feats = predict(model, X['val'])
    temperature = fit_temperature(val_logits, y['val']) if TEMPERATURE_SCALE else 1.0
    test_logits, test_feats = predict(model, X['test'])
    tm, test_probs = compute_all_metrics(test_logits, y['test'], temperature)
    wall = time.perf_counter() - start

    # ---- reliability: does the score flag the unseen magnification (test vs val) and another institution? ----
    _, train_feats = predict(model, X['train'])
    logits = {'val': val_logits, 'test': test_logits}
    feats = {'val': val_feats, 'test': test_feats}
    if BACH_TOKENS is not None:
        logits['bach'], feats['bach'] = predict(model, BACH_TOKENS[:, zsel][:, :, token_slice])
    means, prec = fit_mahalanobis(train_feats, y['train'])
    scores = {'maha': {s: mahalanobis_score(f, means, prec) for s, f in feats.items()},
              'knn': {s: knn_score(f, train_feats) for s, f in feats.items()},
              'msp': {s: neg_msp(l) for s, l in logits.items()},
              'energy': {s: energy_from_logits(l) for s, l in logits.items()}}
    ood = {f'{score}_{pair}_auroc': ood_auroc(sc[a], sc[b])
           for score, sc in scores.items() for pair, (a, b) in OOD_PAIRS.items() if a in sc and b in sc}

    run_id = run_id_for(variant, ho, seed)
    run_dir = Path(OUTPUT, 'runs', run_id); run_dir.mkdir(parents=True, exist_ok=True)
    record = {'run_id': run_id, 'branch': 'magscan_pilot', 'model': f'magscan:{variant}', 'variant': variant,
              'held_out_mag': ho, 'patient_fraction': PATIENT_FRACTION, 'seed': seed,
              'n_train': int(n), 'n_val': len(y['val']), 'n_test': len(y['test']), 'best_epoch': best_epoch,
              'trainable_params': int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
              'peak_memory_mib': torch.cuda.max_memory_allocated() / 2**20, 'wallclock_s': wall,
              'split_check': benchmark_split_check(ho, seed, idx['test']), 'config_hash': CONFIG_HASH,
              'scan_backend': SCAN_BACKEND, **{f'test_{k}': v for k, v in tm.items()}, **ood}
    pd.DataFrame(log).to_csv(run_dir / 'epochs.csv', index=False)
    np.savez_compressed(run_dir / 'test_outputs.npz', logits=test_logits, labels=y['test'], probs=test_probs,
                        manifest_index=idx['test'])
    torch.save({'state_dict': {k: v.cpu() for k, v in best_state.items()}, 'variant': variant,
                'temperature': temperature}, run_dir / 'head.pt')
    (run_dir / 'record.json').write_text(json.dumps(record, indent=2, default=str))       # written last = finished
    del model; torch.cuda.empty_cache()
    return record


print('config hash', CONFIG_HASH)

In [ ]:
# ---------- Run every (held-out magnification, seed, variant); resumable ----------
# A run is finished when runs/<run_id>/record.json exists. Finished runs of an attached earlier version of this
# notebook are restored first, but only if they were produced with the same configuration (config hash).
if Path('/kaggle/input').exists():
    _restored = 0
    for rec in Path('/kaggle/input').rglob('record.json'):
        if rec.parent.parent.name != 'runs' or not rec.parent.name.startswith('magscan_'):
            continue
        try:
            r = json.loads(rec.read_text())
        except Exception:
            continue
        dst = Path(OUTPUT, 'runs', rec.parent.name)
        if r.get('config_hash') == CONFIG_HASH and not (dst / 'record.json').exists():
            shutil.copytree(rec.parent, dst, dirs_exist_ok=True); _restored += 1
    print(f'restored {_restored} finished run(s) from attached earlier versions')

# ---- split check first: stop before any GPU time is spent if a split differs from the benchmark's ----
_checks = {(ho, seed): benchmark_split_check(ho, seed, split_indices(ho, seed)['test'])
           for ho in HELD_OUT_MAGS for seed in SEEDS}
for (ho, seed), status in _checks.items():
    print(f'  split check held-out {ho}x seed {seed}: {status}')
assert 'DIFFERS' not in _checks.values(), 'a test split differs from the benchmark: stop and check the manifest'

try:
    import psutil
    KERNEL_START = psutil.Process(os.getpid()).create_time()
except Exception:
    KERNEL_START = time.time()
deadline = KERNEL_START + (MAX_RUNTIME_HOURS - 0.3) * 3600

planned = [(ho, seed, v) for ho in HELD_OUT_MAGS for seed in SEEDS for v in VARIANTS]
records, longest = [], {}
for i, (ho, seed, v) in enumerate(planned, 1):
    rid = run_id_for(v, ho, seed)
    f = Path(OUTPUT, 'runs', rid, 'record.json')
    if f.exists():
        records.append(json.loads(f.read_text())); continue
    need = 1.2 * longest.get(v, 300.0)
    if time.time() + need > deadline:
        print(f'[{i}/{len(planned)}] not started (time budget): {rid}'); continue
    t0 = time.perf_counter()
    try:
        rec = run_variant(v, ho, seed)
        records.append(rec)
        longest[v] = max(longest.get(v, 0.0), time.perf_counter() - t0)
        print(f'[{i}/{len(planned)}] {rid:<42} F1 {rec["test_macro_f1"]:.3f}  ECE {rec["test_ece"]:.3f}  '
              f'risk@80 {rec["test_risk_at_coverage_80"]:.3f}  maha test-vs-val {rec["maha_test_vs_val_auroc"]:.3f}  '
              f'({(time.perf_counter() - t0) / 60:.1f} min, split {rec["split_check"]})')
    except Exception as exc:
        import traceback; traceback.print_exc()
        print(f'[{i}/{len(planned)}] FAILED {rid}: {repr(exc)[:300]}')

PILOT = pd.DataFrame(records)
if len(PILOT):
    PILOT.to_csv(Path(OUTPUT, 'pilot_runs.csv'), index=False)
    bad = PILOT[PILOT.split_check == 'DIFFERS']
    assert bad.empty, f'test labels differ from the benchmark for {bad.run_id.tolist()}'
print(f'{len(PILOT)} of {len(planned)} runs finished')

In [ ]:
# ---------- Pilot result: the variants side by side, the benchmark's encoders as reference, and the verdict ----------
import matplotlib.pyplot as plt
import zipfile

DECISION_RULE = """Decision rule (fixed before the results):
  (a) accuracy     MagScan-Mamba's mean macro-F1 is higher than cls_probe's at every held-out magnification
  (b) reliability  its mean ECE and risk at 80% coverage are at most 0.01 above cls_probe's at every one
  (c) the scan     it beats noscan and attn on mean macro-F1 at every held-out magnification (state-space claim)
  (d) the zooms    it beats magscan_single on mean macro-F1 at every held-out magnification (zoom-pyramid claim)
  MagScan-Mamba passes the pilot if (a) and (b) hold; (c) and (d) decide which claims the paper can make."""
print(DECISION_RULE)

METRICS = [('test_macro_f1', 'Macro-F1', True), ('test_balanced_accuracy', 'Bal. acc.', True),
           ('test_ece', 'ECE', False), ('test_brier', 'Brier', False),
           ('test_risk_at_coverage_80', 'Risk@80', False), ('test_risk_coverage_auc', 'AURC', False),
           ('test_e_aurc', 'E-AURC', False), ('maha_test_vs_val_auroc', 'Maha AUROC test-vs-val', True),
           ('maha_bach_vs_test_auroc', 'Maha AUROC BACH-vs-test', True),
           ('msp_bach_vs_test_auroc', 'MSP AUROC BACH-vs-test', True)]
if not len(PILOT):
    print('No finished runs.')
else:
    cols = [c for c, _, _ in METRICS if c in PILOT.columns]
    order = [v for v in VARIANTS if v in set(PILOT.variant)]
    summ = PILOT.groupby(['held_out_mag', 'variant'])[cols].agg(['mean', 'std'])
    summ.to_csv(Path(OUTPUT, 'pilot_summary.csv'))
    pd.set_option('display.width', 250)
    for ho in HELD_OUT_MAGS:
        if ho not in summ.index.get_level_values(0):
            continue
        t = summ.loc[ho].reindex(order)
        show = pd.DataFrame({lab: t[(c, 'mean')].map('{:.3f}'.format) + ' ± ' + t[(c, 'std')].fillna(0).map('{:.3f}'.format)
                             for c, lab, _ in METRICS if c in cols}, index=t.index)
        print(f'\n=== held-out {ho}x: mean ± SD over seeds ===')
        print(show.to_string())

    # ---- the benchmark's encoders at the same held-out magnifications (attached outputs) ----
    _ref = []
    for d in BENCHMARK_DIRS:
        for rec in Path(d, 'runs').glob('*/record.json'):
            r = json.loads(rec.read_text())
            if (r.get('branch') in ('from_scratch', 'fm_linear_probe') and r.get('held_out_mag') in HELD_OUT_MAGS
                    and float(r.get('patient_fraction', 1.0)) == PATIENT_FRACTION and r.get('seed') in SEEDS):
                _ref.append(r)
    REF = pd.DataFrame(_ref).drop_duplicates('run_id') if _ref else pd.DataFrame()
    if len(REF):
        REF['test_e_aurc'] = REF.test_risk_coverage_auc - (lambda e: e + (1 - e) * np.log(np.clip(1 - e, 1e-12, None)))(1 - REF.test_accuracy)
        _rc = [c for c in cols if c in REF.columns]
        ref_summ = REF.groupby(['held_out_mag', 'model'])[_rc].agg(['mean', 'std'])
        ref_summ.to_csv(Path(OUTPUT, 'benchmark_reference.csv'))
        print('\n=== benchmark encoders at the same held-out magnifications and seeds (attached outputs) ===')
        print(REF.groupby(['held_out_mag', 'model'])[_rc].mean().round(3).to_string())
    else:
        print('\n(benchmark outputs not attached: no reference rows)')

    # ---- paired differences: MagScan-Mamba minus each other variant, same seed ----
    drows = []
    if 'magscan' in order:
        for ho in HELD_OUT_MAGS:
            base = PILOT[(PILOT.held_out_mag == ho) & (PILOT.variant == 'magscan')].set_index('seed')
            for v in order:
                if v == 'magscan':
                    continue
                other = PILOT[(PILOT.held_out_mag == ho) & (PILOT.variant == v)].set_index('seed')
                common = base.index.intersection(other.index)
                for c, lab, higher in METRICS[:5]:
                    if c not in cols or not len(common):
                        continue
                    diff = base.loc[common, c] - other.loc[common, c]
                    drows.append({'held_out_mag': ho, 'versus': v, 'metric': lab, 'n_seeds': len(common),
                                  'delta_mean': diff.mean(), 'delta_sd': diff.std(),
                                  'seeds_magscan_better': int(((diff > 0) if higher else (diff < 0)).sum())})
    DELTAS = pd.DataFrame(drows)
    if len(DELTAS):
        DELTAS.to_csv(Path(OUTPUT, 'pilot_paired_deltas.csv'), index=False)
        print('\n=== MagScan-Mamba minus each variant (paired by seed; positive delta = higher value) ===')
        print(DELTAS.pivot_table(index=['held_out_mag', 'versus'], columns='metric', values='delta_mean')
              .round(3).to_string())

    # ---- verdict ----
    def _m(ho, v, c):
        s = PILOT[(PILOT.held_out_mag == ho) & (PILOT.variant == v)][c]
        return s.mean() if len(s) else np.nan
    mags = [ho for ho in HELD_OUT_MAGS if not np.isnan(_m(ho, 'magscan', 'test_macro_f1'))]
    def _all(pred):
        vals = [pred(ho) for ho in mags]
        return bool(vals) and all(bool(v) for v in vals)          # a missing variant (NaN) fails the test
    verdict = {
        '(a) accuracy vs cls_probe': _all(lambda h: _m(h, 'magscan', 'test_macro_f1') > _m(h, 'cls_probe', 'test_macro_f1')),
        '(b) reliability not worse than cls_probe': _all(lambda h: (_m(h, 'magscan', 'test_ece') <= _m(h, 'cls_probe', 'test_ece') + 0.01)
                                                         and (_m(h, 'magscan', 'test_risk_at_coverage_80')
                                                              <= _m(h, 'cls_probe', 'test_risk_at_coverage_80') + 0.01)),
        '(c) scan beats noscan and attn': _all(lambda h: _m(h, 'magscan', 'test_macro_f1') > max(_m(h, 'noscan', 'test_macro_f1'),
                                                                                             _m(h, 'attn', 'test_macro_f1'))),
        '(d) zooms beat single view': _all(lambda h: _m(h, 'magscan', 'test_macro_f1') > _m(h, 'magscan_single', 'test_macro_f1'))}
    if len(REF):
        best_ref = REF.groupby(['held_out_mag', 'model']).test_macro_f1.mean().groupby(level=0).max()
        verdict['(extra) beats the best benchmark encoder on macro-F1'] = _all(
            lambda h: h in best_ref.index and _m(h, 'magscan', 'test_macro_f1') > best_ref[h])
    if not mags:
        passed = False
        print('\n=== VERDICT: not possible - no finished magscan run yet (see the protocol cell) ===')
    else:
        print('\n=== VERDICT (held-out ' + ', '.join(f'{h}x' for h in mags) + ') ===')
        missing = [h for h in HELD_OUT_MAGS if h not in mags]
        if missing:
            print(f'  (no magscan runs at held-out {missing}: the verdict covers only the magnifications above)')
        for k, v in verdict.items():
            print(f'  {"PASS" if v else "fail"}  {k}')
        passed = verdict['(a) accuracy vs cls_probe'] and verdict['(b) reliability not worse than cls_probe']
        print('\n-> ' + ('MagScan-Mamba passes the pilot: run the full study.' if passed else
                         'MagScan-Mamba does NOT pass the pilot as specified: revise before the full study.'))
    Path(OUTPUT, 'pilot_verdict.json').write_text(json.dumps({'rule': DECISION_RULE, 'verdict': verdict,
                                                              'passed': passed}, indent=2))

    # ---- figure: every seed of every variant, with the benchmark encoders as dashed lines ----
    fig_metrics = [m for m in METRICS if m[0] in ('test_macro_f1', 'test_ece', 'test_risk_at_coverage_80',
                                                  'maha_test_vs_val_auroc') and m[0] in cols]
    fig_mags = [ho for ho in HELD_OUT_MAGS if (PILOT.held_out_mag == ho).any()]    # every variant, not only magscan
    fig, axes = plt.subplots(len(fig_mags), len(fig_metrics), figsize=(4.2 * len(fig_metrics), 3.6 * len(fig_mags)),
                             squeeze=False)
    colors = {'cls_probe': '#D55E00', 'cls3_probe': '#E69F00', 'noscan': '#999999', 'attn': '#56B4E9',
              'magscan_single': '#CC79A7', 'magscan': '#7B2D8E'}
    ref_colors = {'convnext_tiny': '#0072B2', 'swin_tiny': '#009E73', 'medmamba': '#CC79A7', 'owkin/phikon': '#D55E00'}
    for r, ho in enumerate(fig_mags):
        for c, (col, lab, higher) in enumerate(fig_metrics):
            ax = axes[r, c]
            for k, v in enumerate(order):
                vals = PILOT[(PILOT.held_out_mag == ho) & (PILOT.variant == v)][col]
                ax.bar(k, vals.mean(), color=colors.get(v, '#666666'), alpha=0.35 if v != 'magscan' else 0.8,
                       edgecolor='black', linewidth=0.6)
                ax.scatter([k] * len(vals), vals, color='black', s=12, zorder=3)
            ref_vals = []
            if len(REF) and col in REF.columns:
                for mdl, g in REF[REF.held_out_mag == ho].groupby('model'):
                    ax.axhline(g[col].mean(), color=ref_colors.get(mdl, '#444444'), linestyle='--', linewidth=1.1,
                               label=f'benchmark {mdl}')
                    ref_vals.append(g[col].mean())
            ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=35, ha='right', fontsize=8)
            ax.set_title(f"held-out {ho}x: {lab} ({'higher' if higher else 'lower'} is better)", fontsize=9.5)
            ax.grid(axis='y', alpha=0.3)
            ax.ticklabel_format(axis='y', useOffset=False)
            vals_all = pd.concat([PILOT[PILOT.held_out_mag == ho][col], pd.Series(ref_vals, dtype=float)]).dropna()
            if len(vals_all):                                  # the range covers the reference lines too
                lo, hi = vals_all.min(), vals_all.max()
                pad = 0.15 * (hi - lo + 1e-3)
                ax.set_ylim(max(0, lo - pad), hi + pad)
    h, l = axes[0, 0].get_legend_handles_labels()
    if h:
        fig.legend(h, l, loc='lower center', ncol=len(h), fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.02))
    fig.suptitle('MagScan-Mamba pilot: variants (bars = mean, dots = seeds); dashed = benchmark encoders',
                 fontsize=11, fontweight='bold')
    fig.tight_layout(rect=(0, 0.04, 1, 0.96))
    fig.savefig(Path(OUTPUT, 'pilot_results.png'), dpi=200, bbox_inches='tight')
    fig.savefig(Path(OUTPUT, 'pilot_results.pdf'), bbox_inches='tight')
    plt.show()

    # ---- one small ZIP with everything needed to review the pilot (no weights) ----
    with zipfile.ZipFile(Path(OUTPUT, 'pilot_results.zip'), 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in Path(OUTPUT).rglob('*'):
            if p.is_file() and p.suffix in ('.csv', '.json', '.png', '.pdf') and p.name != 'pilot_results.zip':
                zf.write(p, p.relative_to(OUTPUT))
    print(f'\nDownload {OUTPUT}/pilot_results.zip (records, tables, figure; no weights).')